# 07 — HDR by the Linear Digital Image Composer (LDIC)

Paper Section 3.3.2 (Druckmüllerová 2014) — see `docs/LDIC.md` for a full
description.  Weighted average of the nine exposures with intensity‑dependent
weights (0 below 2 %, ramp to 1 at 15 %, ramp down 65→80 %, bypassed at the
ends of the bracket), per‑exposure linear gain $k_i(\phi)$ fitted in 60 angular
sectors through the origin ($q_i=0$) and smoothed with a 4th‑order trigonometric
polynomial.  The weight function is evaluated on a **master luminance** (max of
the three polariser channels) so all channels hand over at the same pixels.

Output: `products/hdr/hdr_ldic_pol{1,2,3}.fits`, `products/ldic_diagnostics.npz`,
`figures/ldic_weights_and_gains.png`.
Legacy source: `make_ldic_hdr_from_stacked_exposures_Chaitanya.ipynb`.  ⏱ ~5 min.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
from scipy import ndimage

def load_plane(inv_exp, i, pos):
    d = np.asarray(fits.getdata(utils.stacked_filename(inv_exp), memmap=True)[i], dtype=np.float32)
    return ndimage.shift(d, config.CHANNEL_SHIFTS[pos], order=1) if any(config.CHANNEL_SHIFTS[pos]) else d

# Pass 1: one global scale per channel (prepare_images_for_ldic semantics)
gmax = {pos: max(float(np.clip(load_plane(e, i, pos), 0, None).max()) for e in config.INV_EXPOSURES)
        for i, pos in enumerate(config.POLARIZER_POSITIONS)}
scale = {pos: config.LDIC_TARGET_MAX / gmax[pos] for pos in gmax}
print("global max per channel:", {k: round(v, 4) for k, v in gmax.items()})

In [ ]:
# Pass 2: master luminance reference per exposure = max over the three scaled channels
refs = []
for e in config.INV_EXPOSURES:
    r = None
    for i, pos in enumerate(config.POLARIZER_POSITIONS):
        p = np.clip(np.clip(load_plane(e, i, pos), 0, None) * scale[pos], 0, config.LDIC_TARGET_MAX).astype(np.float32)
        r = p if r is None else np.maximum(r, p)
    refs.append(r)
print(len(refs), "reference images")

In [ ]:
# Exposure-ratio diagnostic (max of short / max of next longer; 0.5 per doubling = linear, ~1 = saturated)
ratios = {}
for i, pos in enumerate(config.POLARIZER_POSITIONS):
    planes = {e: load_plane(e, i, pos) for e in config.INV_EXPOSURES}
    ratios[pos] = utils.exposure_ratio_table(planes).set_index("step")["ratio"]
ratio_table = pd.DataFrame(ratios); ratio_table["expected"] = utils.exposure_ratio_table(planes).set_index("step")["expected"]
ratio_table.to_csv(config.PRODUCTS_DIR / "exposure_ratio_table.csv")
ratio_table.round(3)

In [ ]:
sun_cx, sun_cy = config.SUN_CENTER_XY
template = fits.getheader(utils.stacked_filename(config.INV_EXPOSURES[0]))
hdr_planes, diagnostics = {}, {}
for i, pos in enumerate(config.POLARIZER_POSITIONS):
    t0 = time.time()
    imgs = [np.clip(np.clip(load_plane(e, i, pos), 0, None) * scale[pos], 0, config.LDIC_TARGET_MAX).astype(np.float32)
            for e in config.INV_EXPOSURES]
    hdr_planes[pos], diagnostics[pos] = utils.ldic_hdr_stacking(
        imgs, config.EXPOSURE_TIMES_S, images_ref=refs, max_pixel_value_input=config.LDIC_TARGET_MAX,
        sun_center_x=sun_cx, sun_center_y=sun_cy, verbose=True, return_diagnostics=True, **config.LDIC_PARAMS)
    del imgs
    fits.writeto(utils.hdr_filename("ldic", pos), hdr_planes[pos], header=utils.hdr_header(template, pos, "ldic"), overwrite=True)
    print(f"pol{pos}: min {hdr_planes[pos].min():.4g} max {hdr_planes[pos].max():.4g}  ({time.time()-t0:.0f} s)")
np.savez(config.PRODUCTS_DIR / "ldic_diagnostics.npz",
         **{f"k_at_pol{pos}": np.array([d["k_at"] for d in diagnostics[pos]]) for pos in diagnostics},
         **{f"weight_fraction_pol{pos}": np.array([d["weight_fraction"] for d in diagnostics[pos]]) for pos in diagnostics},
         **{f"n_valid_pol{pos}": np.array([d["n_valid_segments"] for d in diagnostics[pos]]) for pos in diagnostics},
         exposure_times_sorted=np.array([d["exposure_time"] for d in diagnostics["1"]]),
         segment_angles=diagnostics["1"][0]["segment_angles"])

In [ ]:
# Diagnostics: the weight function and the fitted gains k_i(phi) per exposure (used by docs/LDIC.md)
P = config.LDIC_PARAMS; M = config.LDIC_TARGET_MAX
x = np.linspace(0, M, 1000)
w_mid = utils.weight_function_ldic(x, P["wf_low_reject_percent"]/100*M, P["wf_low_full_percent"]/100*M,
                                   P["wf_high_start_reject_percent"]/100*M, P["wf_high_reject_percent"]/100*M)
w_long = utils.weight_function_ldic(x, 0, 0, P["wf_high_start_reject_percent"]/100*M, P["wf_high_reject_percent"]/100*M)
w_short = utils.weight_function_ldic(x, P["wf_low_reject_percent"]/100*M, P["wf_low_full_percent"]/100*M, M, M)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(x / M * 100, w_mid, lw=2, label="intermediate exposures")
ax[0].plot(x / M * 100, w_long, lw=1.5, ls="--", label="longest exposure (low-end bypass)")
ax[0].plot(x / M * 100, w_short, lw=1.5, ls=":", label="shortest exposure (high-end bypass)")
ax[0].set_xlabel("master-luminance pixel value [% of dynamic range]"); ax[0].set_ylabel("weight w"); ax[0].legend(fontsize=9); ax[0].grid(alpha=0.3)
ax[0].set_title("LDIC weight function")
exps = np.load(config.PRODUCTS_DIR / "ldic_diagnostics.npz")
ang = np.degrees(exps["segment_angles"])
for pos, col in zip(config.POLARIZER_POSITIONS, ("r", "g", "b")):
    k = exps[f"k_at_pol{pos}"]
    for j in range(1, len(k)):
        ax[1].plot(ang, k[j], color=col, alpha=0.35 + 0.65 * j / len(k), lw=1)
ax[1].set_yscale("log"); ax[1].set_xlabel("position angle φ [deg, image frame]"); ax[1].set_ylabel("gain k_i(φ)")
ax[1].set_title("fitted gains per exposure (colour = polariser; darker = shorter exposure)"); ax[1].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(config.FIGURES_DIR / "ldic_weights_and_gains.png", dpi=150, bbox_inches="tight")
print("mean gain per exposure step (pol1):", np.round([float(np.mean(k)) for k in exps["k_at_pol1"]], 3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, pos in zip(axes, config.POLARIZER_POSITIONS):
    ax.imshow(utils.asinh_stretch(hdr_planes[pos][::4, ::4]), cmap="gray"); ax.set_title(f"LDIC HDR pol{pos} (asinh)"); ax.axis("off")
plt.tight_layout()

In [ ]:
from PIL import Image
rgb = np.stack([utils.normalise_channel(hdr_planes[p]) for p in config.POLARIZER_POSITIONS], axis=-1)
Image.fromarray((rgb * 255).astype(np.uint8)).save(config.HDR_DIR / "hdr_ldic_preview.png")
plt.figure(figsize=(8, 5.5)); plt.imshow(rgb[::4, ::4]); plt.axis("off"); plt.title("LDIC HDR preview")